# Cách 2. Colab T4 with RAG & FineTunne LiquidAI/LFM2.5-1.2B-Thinking

In [ ]:
%%capture
# 1. CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
# Phiên bản unsloth mới nhất hỗ trợ tốt LFM2.5
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Cài đặt các thư viện hỗ trợ 4-bit quantization và training
!pip install --no-deps "xformers<0.0.26" triton peft accelerate bitsandbytes
# Cài đặt llama-cpp-python để hỗ trợ export GGUF
!pip install llama-cpp-python

# Tắt wandb để tránh đăng nhập không cần thiết
import os
os.environ["WANDB_DISABLED"] = "true"

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig

# 2. LOAD MODEL LFM2.5-1.2B-THINKING
# Sử dụng dtype=None để tự động chọn bf16 hoặc fp16 tương thích với T4
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "thang/iqw2.5-7b_kai-wf_cardvn2025", 
    max_seq_length = 2048,  # Có thể tăng lên 4096 nếu cần context dài, nhưng 2048 sẽ train nhanh hơn
    dtype = None, 
    load_in_4bit = True,    # Tải dưới dạng 4-bit để tiết kiệm VRAM (dù 1.2B đã rất nhẹ)
)

# 3. CẤU HÌNH LORA CHO LFM2.5
# Lưu ý: LFM2.5 có kiến trúc hơi khác Llama, đặc biệt là module 'in_proj'
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"], # Danh sách chuẩn cho LFM2.5
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Quan trọng cho T4 để tiết kiệm bộ nhớ
    random_state = 3407,
    use_rslora = True, # Ranked LoRA giúp ổn định hơn
)

# 4. LOAD DỮ LIỆU (Audit Card 2024 & 2025 & KNIME K-AI, PBI report)
print("Đang tải dữ liệu training...")
try:
    # Load cả 2 file jsonl
    dataset = load_dataset(
        "json", 
        data_files=["audit-card2024.jsonl", "audit-card2025.jsonl", "k-ai2025.jsonl"], 
        split="train"
    )
    
    # Chuẩn bị định dạng ShareGPT cho Unsloth
    from unsloth.chat_templates import standardize_sharegpt
    dataset = standardize_sharegpt(dataset)
    
    # Hàm định dạng prompt sử dụng Chat Template mặc định của LFM2.5 (ChatML-like)
    def formatting_prompts_func(examples):
        convos = examples["messages"]
        # LFM2.5 dùng ChatML, apply_chat_template sẽ tự xử lý đúng định dạng
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts, }
    
    dataset = dataset.map(formatting_prompts_func, batched = True)
    print(f"✅ Đã tải dữ liệu thành công! Tổng số mẫu: {len(dataset)}")

except Exception as e:
    print(f"⚠️ Lỗi khi tải file: {e}")
    print("Đang sử dụng dữ liệu giả (dummy data) để test cấu hình...")
    
    # Dữ liệu giả định dạng giống thật để test luồng
    dummy_data = [
        {
            "messages": [
                {"role": "system", "content": "Bạn là chuyên gia tư vấn audit thẻ ngân hàng."}, 
                {"role": "user", "content": "Tại sao cần kiểm tra inactive accounts?"}, 
                {"role": "assistant", "content": "Để giảm thiểu rủi ro bảo mật, gian lận và tuân thủ quy định Ngân hàng."}
            ]
        }
    ]
    dataset = Dataset.from_list(dummy_data)
    
    from unsloth.chat_templates import standardize_sharegpt
    dataset = standardize_sharegpt(dataset)
    
    def formatting_prompts_func(examples):
        convos = examples["messages"]
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts, }
    
    dataset = dataset.map(formatting_prompts_func, batched = True)

# 5. TRAINING (SFT)
# Với model 1.2B, bạn có thể tăng max_steps hoặc num_train_epochs tùy ý mà không lo quá tải
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2, # T4 15GB có thể nhồi lên tới 4, nhưng 2 là an toàn
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, 
        # Nếu dataset của bạn lớn (hàng ngàn mẫu), hãy đổi max_steps thành: num_train_epochs = 1 hoặc 2
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        dataset_text_field = "text", # Quan trọng: trỏ đến trường text đã format
    ),
)
print("🚀 Bắt đầu training...")
trainer.train()

# 6. LƯU MODEL (LoRA adapters)
print("💾 Đang lưu LoRA adapters...")
model.save_pretrained("lfm25_thinking_lora") 
tokenizer.save_pretrained("lfm25_thinking_lora")

# 7. EXPORT GGUF (Để chạy trên LM Studio/Ollama)
print("🔄 Bắt đầu export GGUF (đây là bước mất thời gian nhất)...")
# Unsloth sẽ tự merge LoRA vào base model rồi quantize
model.save_pretrained_gguf(
    "lfm25_thinking_gguf", 
    tokenizer, 
    quantization_method = "q4_k_m" # Cân bằng giữa tốc độ và độ chính xác
)
print("✅ XONG! File GGUF đã nằm trong thư mục: /content/lfm25_thinking_gguf")

Đoạn Code Copy sang Google Drive (Đã cập nhật tên thư mục)

Đoạn code này tương tự nhưng tôi đã cập nhật đường dẫn nguồn và đích để khớp với tên model mới:

In [ ]:
from google.colab import drive
import os

# 1. Kết nối với Google Drive
drive.mount('/content/drive')

# 2. Định nghĩa đường dẫn
# Nguồn: Thư mục chứa file GGUF vừa tạo
source_dir = "/content/lfm25_thinking_gguf"

# Đích: Thư mục trên Google Drive (Đã đổi tên để phân biệt với model cũ)
target_dir = "/content/drive/MyDrive/AI_Models_LFM25_Thinking_Audit"

# 3. Tạo thư mục đích nếu chưa có
os.makedirs(target_dir, exist_ok=True)

# 4. Thực hiện lệnh Copy
print(f"Đang bắt đầu copy từ {source_dir} sang Google Drive...")
print("⏳ Quá trình này có thể mất từ 2 đến 5 phút vì model 1.2B khá nhẹ.")

# Thực hiện lệnh copy
!cp -r {source_dir} {target_dir}

print("\n>>> HOÀN TẤT! Model LFM2.5-Thinking đã sẵn sàng trong thư mục AI_Models_LFM25_Thinking_Audit trên Google Drive.")

# Sửa lỗi cách 2

In [ ]:
%%capture
# 1. CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
# Phiên bản unsloth mới nhất hỗ trợ tốt LFM2.5
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Cài đặt các thư viện hỗ trợ 4-bit quantization và training
!pip install --no-deps "xformers<0.0.26" triton peft accelerate bitsandbytes
# Cài đặt llama-cpp-python để hỗ trợ export GGUF
!pip install llama-cpp-python

# Tắt wandb để tránh đăng nhập không cần thiết
import os
os.environ["WANDB_DISABLED"] = "true"

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig

# 2. LOAD MODEL LFM2.5-1.2B-THINKING
# Sử dụng dtype=None để tự động chọn bf16 hoặc fp16 tương thích với T4
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "LiquidAI/LFM2.5-1.2B-Thinking", 
    max_seq_length = 2048,  # Có thể tăng lên 4096 nếu cần context dài, nhưng 2048 sẽ train nhanh hơn
    dtype = None, 
    load_in_4bit = True,    # Tải dưới dạng 4-bit để tiết kiệm VRAM (dù 1.2B đã rất nhẹ)
)

# 3. CẤU HÌNH LORA CHO LFM2.5
# Lưu ý: LFM2.5 có kiến trúc hơi khác Llama, đặc biệt là module 'in_proj'
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"], # Danh sách chuẩn cho LFM2.5
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Quan trọng cho T4 để tiết kiệm bộ nhớ
    random_state = 3407,
    use_rslora = True, # Ranked LoRA giúp ổn định hơn
)

# 4. LOAD DỮ LIỆU (ĐÃ SỬA)
print("Đang tải dữ liệu training...")
try:
    dataset = load_dataset(
        "json",
        data_files=["audit-card2024.jsonl", "audit-card2025.jsonl", "k-ai2025.jsonl"],
        split="train"
    )
    
    # KHÔNG DÙNG standardize_sharegpt vì data đã đúng format OpenAI {"messages": ...}
    # Truy cập trực tiếp vào cột "messages"
    
    # Hàm định dạng prompt sử dụng Chat Template của LFM2.5
    def formatting_prompts_func(examples):
        convos = examples["messages"]
        # LFM2.5 dùng ChatML, add_generation_prompt=False vì đây là dữ liệu huấn luyện (đã có đầy đủ câu trả lời)
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts, }

    dataset = dataset.map(formatting_prompts_func, batched = True)
    
    # In thử 1 mẫu để kiểm tra xem template đã đúng chưa
    print("Ví dụ dữ liệu sau khi format:")
    print(dataset[0]['text'][:200]) # In 200 ký tự đầu
    
    print(f"✅ Đã tải dữ liệu thành công! Tổng số mẫu: {len(dataset)}")

except Exception as e:
    print(f"⚠️ Lỗi khi tải file: {e}")
    raise # Dừng chạy nếu lỗi để sửa ngay

# 5. TRAINING (ĐÃ SỬA - Tăng epoch)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2, 
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        # --- SỬA ĐỔI QUAN TRỌNG ---
        # Thay vì max_steps=60, ta dùng num_train_epochs để đảm bảo học hết dữ liệu
        num_train_epochs = 2, 
        # -------------------------
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        dataset_text_field = "text", 
    ),
)
print("🚀 Bắt đầu training (khoảng 5-10 phút với model 1.2B)...")
trainer.train()

# 6. LƯU MODEL (LoRA adapters)
print("💾 Đang lưu LoRA adapters...")
model.save_pretrained("lfm25_thinking_lora") 
tokenizer.save_pretrained("lfm25_thinking_lora")

# 7. EXPORT GGUF (Để chạy trên LM Studio/Ollama)
print("🔄 Bắt đầu export GGUF (đây là bước mất thời gian nhất)...")
# Unsloth sẽ tự merge LoRA vào base model rồi quantize
model.save_pretrained_gguf(
    "lfm25_thinking_gguf", 
    tokenizer, 
    quantization_method = "q4_k_m" # Cân bằng giữa tốc độ và độ chính xác
)
print("✅ XONG! File GGUF đã nằm trong thư mục: /content/lfm25_thinking_gguf")

In [ ]:
from google.colab import drive
import os

# 1. Kết nối với Google Drive
drive.mount('/content/drive')

# 2. Định nghĩa đường dẫn
# Nguồn: Thư mục chứa file GGUF vừa tạo
source_dir = "/content/lfm25_thinking_gguf"

# Đích: Thư mục trên Google Drive (Đã đổi tên để phân biệt với model cũ)
target_dir = "/content/drive/MyDrive/AI_Models_LFM25_Thinking_Audit"

# 3. Tạo thư mục đích nếu chưa có
os.makedirs(target_dir, exist_ok=True)

# 4. Thực hiện lệnh Copy
print(f"Đang bắt đầu copy từ {source_dir} sang Google Drive...")
print("⏳ Quá trình này có thể mất từ 2 đến 5 phút vì model 1.2B khá nhẹ.")

# Thực hiện lệnh copy
!cp -r {source_dir} {target_dir}

print("\n>>> HOÀN TẤT! Model LFM2.5-Thinking đã sẵn sàng trong thư mục AI_Models_LFM25_Thinking_Audit trên Google Drive.")

# Colab T4 with RAG & FineTunne Qwen/Qwen2.5-7B-Instruct

có một số điểm quan trọng cần làm rõ trước khi sửa code:

Về môi trường chạy: Đoạn code bạn cung cấp sử dụng thư viện unsloth, bitsandbytes và xformers. Những thư viện này bắt buộc cần GPU NVIDIA (CUDA) để hoạt động. Bạn không thể chạy đoạn code huấn luyện (training) này trên laptop có VGA Intel HD 620 được. Để chạy code này, bạn cần sử dụng Google Colab (với GPU T4 miễn phí) hoặc các dịch vụ Cloud GPU.
Về Input Model: Bạn yêu cầu dùng model Qwen/Qwen2.5-7B-Instruct-GGUF. Thực tế, chúng ta không thể "train" (fine-tune) trực tiếp trên file GGUF. Chúng ta phải tải model gốc (định dạng PyTorch/HuggingFace) về để train, sau đó mới xuất (export) ra file GGUF để dùng trên laptop của bạn.
Dưới đây là đoạn code đã được sửa đổi hoàn chỉnh để:

Sử dụng model Qwen/Qwen2.5-7B-Instruct.
Tương thích để train trên Google Colab (GPU T4).
Export ra file GGUF để chạy mượt mà trên laptop Intel của bạn.

In [ ]:
# Code Colab T4: Jupyternotebook:

%%capture
# 1. CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
# Unsloth mới nhất đã hỗ trợ tốt Qwen2.5
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.26" triton peft accelerate bitsandbytes
# Cài đặt llama-cpp-python để hỗ trợ export GGUF
!pip install llama-cpp-python

# Tắt wandb để tránh đăng nhập không cần thiết
import os
os.environ["WANDB_DISABLED"] = "true"

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig

# 2. LOAD MODEL QWEN2.5-7B-INSTRUCT
# Lưu ý: Tải model gốc (PyTorch) để train, không phải GGUF
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct", 
    max_seq_length = 2048,      # 7B nặng hơn 1.2B, 2048 là mức an toàn cho Colab T4
    dtype = None,               # Tự động chọn bf16/fp16
    load_in_4bit = True,        # Bắt buộc dùng 4-bit để vừa VRAM 16GB của T4
)

# 3. CẤU HÌNH LORA CHO QWEN2.5
# Qwen2.5 có kiến trúc Llama-style, các module cần sửa như sau:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    # Các module chuẩn cho Qwen2/Llama3 (Đã sửa bỏ in_proj, thay bằng gate/up/down proj)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", 
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
)

# 4. LOAD DỮ LIỆU
print("Đang tải dữ liệu training...")
try:
    dataset = load_dataset(
        "json",
        data_files=["audit-card2025.jsonl","audit-card2024.jsonl","k-ai2025.jsonl"], # Đảm bảo file này đã được upload
        split="train"
    )

    # Hàm định dạng prompt sử dụng Chat Template của Qwen (ChatML format)
    def formatting_prompts_func(examples):
        convos = examples["messages"]
        # Qwen2.5 dùng ChatML: <|im_start|>user...<|im_end|>
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts }

    dataset = dataset.map(formatting_prompts_func, batched = True)

    print("Ví dụ dữ liệu sau khi format:")
    print(dataset[0]['text'][:300]) 
    print(f"✅ Đã tải dữ liệu thành công! Tổng số mẫu: {len(dataset)}")

except Exception as e:
    print(f"⚠️ Lỗi khi tải file: {e}")
    raise

# 5. TRAINING
# Qwen2.5 7B nặng hơn, nên giảm batch size để tránh tràn bộ nhớ (OOM)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,      # Giảm từ 2 xuống 1 cho 7B model
        gradient_accumulation_steps = 4,      # Tăng bước tích lũy để bù batch size
        warmup_steps = 5,
        num_train_epochs = 2,                 # 2 epoch là vừa đủ
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        dataset_text_field = "text",
        max_seq_length = 2048,                # Cần set lại trong config
        packing = False,                      # Tắt packing để ổn định hơn
    ),
)
print("🚀 Bắt đầu training model Qwen2.5 7B...")
trainer.train()

# 6. LƯU MODEL (LoRA adapters)
print("💾 Đang lưu LoRA adapters...")
model.save_pretrained("qwen25_7b_lora")
tokenizer.save_pretrained("qwen25_7b_lora")

# 7. EXPORT GGUF (Để chạy trên LAPTOP INTEL của bạn)
print("🔄 Bắt đầu export GGUF...")
model.save_pretrained_gguf(
    "qwen25_7b_finetuned_gguf",
    tokenizer,
    quantization_method = "q4_k_m" # Q4_K_M là tối ưu cho CPU/Laptop
)
print("✅ XONG! File GGUF nằm trong thư mục: /content/qwen25_7b_finetuned_gguf")

## Đoạn Code Copy sang Google Drive (Đã cập nhật tên thư mục)

Đoạn code này tương tự nhưng tôi đã cập nhật đường dẫn nguồn và đích để khớp với tên model mới:

In [ ]:
from google.colab import drive
import os

# 1. Kết nối với Google Drive
drive.mount('/content/drive')

# 2. Định nghĩa đường dẫn
# Nguồn: Thư mục chứa file GGUF vừa tạo
source_dir = "/content/qwen25_7b_finetuned_gguf_gguf"

# Đích: Thư mục trên Google Drive (Đã đổi tên để phân biệt với model cũ)
target_dir = "/content/drive/MyDrive/AI_Models_qwen25_7b_Instruct"

# 3. Tạo thư mục đích nếu chưa có
os.makedirs(target_dir, exist_ok=True)

# 4. Thực hiện lệnh Copy
print(f"Đang bắt đầu copy từ {source_dir} sang Google Drive...")
print("⏳ Quá trình này có thể mất từ 2 đến 5 phút vì model 7B ~ 4.6GB.")

# Thực hiện lệnh copy
!cp -r {source_dir} {target_dir}

print("\n>>> HOÀN TẤT! Model qwen25_7b_Instruct đã sẵn sàng trong thư mục AI_Models_qwen25_7b_Instruct trên Google Drive.")

# Colab T4 with RAG & FineTunne Qwen/Qwen3-VL-8B-Instruct-GGUF  >> ivlq3-8b_kai-pbi-cd-GGUUF

có một số điểm quan trọng cần làm rõ trước khi sửa code:

Về môi trường chạy: Đoạn code bạn cung cấp sử dụng thư viện unsloth, bitsandbytes và xformers. Những thư viện này bắt buộc cần GPU NVIDIA (CUDA) để hoạt động. Bạn không thể chạy đoạn code huấn luyện (training) này trên laptop có VGA Intel HD 620 được. Để chạy code này, bạn cần sử dụng Google Colab (với GPU T4 miễn phí) hoặc các dịch vụ Cloud GPU.
Về Input Model: Bạn yêu cầu dùng model Qwen/Qwen3-VL-8B-Instruct-GGUF. Thực tế, chúng ta không thể "train" (fine-tune) trực tiếp trên file GGUF. Chúng ta phải tải model gốc (định dạng PyTorch/HuggingFace) về để train, sau đó mới xuất (export) ra file GGUF để dùng trên laptop của bạn.
Dưới đây là đoạn code đã được sửa đổi hoàn chỉnh để:

Sử dụng model Qwen/Qwen3-VL-8B-Instruct-GGUF.
Tương thích để train trên Google Colab (GPU T4).
Export ra file GGUF để chạy mượt mà trên laptop Intel của bạn.

In [ ]:
# Code Colab A100/H100: Jupyternotebook:
# --- BƯỚC FIX LỖI THƯ VIỆN (PATCH) ---
# Đoạn code này sẽ sửa trực tiếp file lỗi bên trong thư viện unsloth

import os

# Đường dẫn tới file Python đang gây lỗi
file_path = "/usr/local/lib/python3.12/dist-packages/unsloth/models/rl.py"

# Dùng lệnh sed để tìm và thay thế dòng code gây lỗi
# Chúng ta thay thế dòng: sanitize_logprob = RL_REPLACEMENTS["sanitize_logprob"]
# Bằng: sanitize_logprob = False (Giá trị mặc định để không bị lỗi)
!sed -i 's/sanitize_logprob = RL_REPLACEMENTS\["sanitize_logprob"\]/sanitize_logprob = False # Patched fix for KeyError/g' {file_path}

print("✅ Đã vá lỗi thư viện thành công!")
print("👉 Bây giờ hãy chạy lại đoạn code Training của bạn.")

#chạy 1 lần:
!pip install trl==0.12.0

%%capture
# Cài đặt lại sau khi restart
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "peft" "accelerate" "bitsandbytes"
# Quan trọng: Cập nhật TRL để tránh lỗi import
#!pip install --upgrade trl

# Fix lỗi sanitize_logprob (nếu bản unsloth vẫn chưa update fix)
import os
file_path = "/usr/local/lib/python3.12/dist-packages/unsloth/models/rl.py"
!sed -i 's/sanitize_logprob = RL_REPLACEMENTS\["sanitize_logprob"\]/sanitize_logprob = False # Patched fix for KeyError/g' {file_path}

# Tắt wandb để tránh đăng nhập không cần thiết
import os
os.environ["WANDB_DISABLED"] = "true"

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 1. LOAD MODEL (ĐÃ LÀM SẠCH THAM SỐ)
model_name = "Qwen/Qwen2.5-VL-7B-Instruct" # Changed to the text-only version as per notebook's original intention

print(f"Đang tải model: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,      # 2048 is safe for a 7B text model on T4
    dtype = None,               # Let Unsloth choose optimal dtype
    load_in_4bit = True,
    # device_map = "auto" is not strictly necessary for text-only models of this size with Unsloth
)

# 2. CẤU HÌNH LORA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
)

# 3. LOAD DỮ LIỆU (ĐÃ SỬA LỖI VỚI remove_columns)
print("Đang tải và xử lý dữ liệu...")
try:
    dataset = load_dataset("json", data_files=["k-ai2025.jsonl"], split="train")

    # Hàm chuẩn hóa nội dung (giữ nguyên đoạn bạn đã sửa ở bước trước)
    def normalize_content(messages):
        if not isinstance(messages, list):
            return messages
        new_messages = []
        for msg in messages:
            new_msg = msg.copy()
            content = msg.get("content")
            if isinstance(content, list):
                if len(content) > 0 and isinstance(content[0], dict):
                    new_msg["content"] = content
                else:
                    new_msg["content"] = "\n".join(content)
            elif isinstance(content, str):
                new_msg["content"] = content
            new_messages.append(new_msg)
        return new_messages

    def formatting_prompts_func(examples):
        convos = [normalize_content(msg_list) for msg_list in examples["messages"]]
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts }

    # --- PHẦN QUAN TRỌNG NHẤT ---
    # remove_columns=dataset.column_names sẽ XÓA TOÀN BỘ cột cũ (messages, images, v.v.)
    # và chỉ giữ lại cột "text" mới vừa tạo.
    dataset = dataset.map(
        formatting_prompts_func,
        batched = True,
        remove_columns=dataset.column_names
    )
    # ----------------------------

    print(f"✅ Dữ liệu đã sẵn sàng: {len(dataset)} mẫu")
    print(f"Cột dữ liệu hiện tại: {dataset.column_names}") # Kiểm tra xem chỉ còn cột 'text' không
except Exception as e:
    print(f"Lỗi dữ liệu: {e}")
    import traceback
    traceback.print_exc()

# 4. TRAINING
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    max_seq_length = 2048,
    dataset_text_field = "text",
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        packing = False,

        # --- PHẦN CẦN THÊM VÀO ---
        fp16 = True,      # Bật FP16 vì T4 hỗ trợ tốt định dạng này
        bf16 = False,     # Tắt BF16 vì T4 không hỗ trợ
        # ------------------------
    ),
)

print("🚀 Bắt đầu training...")
trainer.train()

# 5. SAVE & EXPORT
print("💾 Lưu model...")
model.save_pretrained("Qwen2.5-VL-7B_lora")
tokenizer.save_pretrained("Qwen2.5-VL-7B_lora")

print("🔄 Export GGUF...")
model.save_pretrained_gguf("Qwen2.5-VL-7B_gguf", tokenizer, quantization_method = "q4_k_m")
print("✅ Hoàn tất!")

In [ ]:
from google.colab import drive
import os

# 1. Kết nối với Google Drive
drive.mount('/content/drive')

# 2. Định nghĩa đường dẫn
# Nguồn: Thư mục chứa file GGUF vừa tạo
source_dir = "/content/Qwen2.5-VL-7B_gguf_gguf"

# Đích: Thư mục trên Google Drive (Đã đổi tên để phân biệt với model cũ)
target_dir = "/content/drive/MyDrive/Qwen2.5-VL-7B_gguf"

# 3. Tạo thư mục đích nếu chưa có
os.makedirs(target_dir, exist_ok=True)

# 4. Thực hiện lệnh Copy
print(f"Đang bắt đầu copy từ {source_dir} sang Google Drive...")
print("⏳ Quá trình này có thể mất từ 10 đến 15 phút vì model 8B ~ 5.3GB.")

# Thực hiện lệnh copy
!cp -r {source_dir} {target_dir}

print("\n>>> HOÀN TẤT! Model Qwen2.5-VL-7B_gguf đã sẵn sàng trong thư mục Qwen2.5-VL-7B_gguf trên Google Drive.")